# Recipe Gemma: FULL LoRA fine-tune on Kaggle (TPU v5e-8, GPU T4 x2 fallback)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikhil3495/Gemma-Fine-Tuning/blob/main/notebooks/train_kaggle_tpu.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/nikhil3495/Gemma-Fine-Tuning/blob/main/notebooks/train_kaggle_tpu.ipynb)

**Open in Kaggle:** use the badge above, or in Kaggle choose *File > Import Notebook > GitHub* and
paste `https://github.com/nikhil3495/Gemma-Fine-Tuning/blob/main/notebooks/train_kaggle_tpu.ipynb`. Then, under *Settings*, set:
- **Accelerator:** `TPU v5e-8` (preferred) or `GPU T4 x2` (fallback),
- **Internet:** on,
- **Environment:** latest (this project needs Python >= 3.12).

This is the **full run** (`configs/train_full.yaml`): **Gemma 3 1B IT** (`gemma3_instruct_1b`),
**LoRA rank 16**, **sequence length 512**, batch 16, up to 3000 steps with early stopping, bf16
mixed precision. The CPU smoke run (`configs/train_smoke.yaml`, Gemma 3 270M, r=8) is the local
counterpart.

The trainer runs on a single accelerator device (no `keras.distribution` sharding), so on a
v5e-8 it uses one chip. That is enough for 1B + LoRA.

**Secrets:** add them under *Add-ons > Secrets*: `KAGGLE_USERNAME`, `KAGGLE_KEY` (needed for the
Kaggle-hosted Gemma preset), `HF_TOKEN` (optional; needed only for the Hub push) and `GEMINI_API_KEY`
(optional; needed only for the Gemini baseline and judge). The values are never printed.

In [ ]:
import sys
assert sys.version_info >= (3, 12), (
    f"Python {sys.version.split()[0]} found; this project (jax 0.12) needs Python >= 3.12. "
    "Switch the runtime to the latest environment/image and restart."
)
print(sys.version)

In [ ]:
# (1) Credentials from Kaggle Secrets -> os.environ (values are never printed).
import os
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
for _name in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN", "GEMINI_API_KEY"]:
    try:
        _val = _secrets.get_secret(_name)
    except Exception:
        _val = None
    if _val:
        os.environ[_name] = _val
    print(f"{_name}: {'set' if _val else 'MISSING'}")
del _secrets, _val

In [ ]:
# (2) Clone the repo into scratch space (not /kaggle/working, so the data is not saved as output).
%cd /tmp
!rm -rf Gemma-Fine-Tuning && git clone --depth 1 https://github.com/nikhil3495/Gemma-Fine-Tuning
%cd /tmp/Gemma-Fine-Tuning

In [ ]:
# (3) Install. TPU -> jax[tpu] from the libtpu index; GPU -> jax[cuda12].
import os, shutil

ACCEL = "tpu" if (os.path.exists("/dev/accel0") or os.environ.get("TPU_ACCELERATOR_TYPE")) else (
    "gpu" if shutil.which("nvidia-smi") else "cpu")
print("accelerator:", ACCEL)

!pip install -q -e ".[train,eval]" safetensors sentencepiece
if ACCEL == "tpu":
    !pip install -q "jax[tpu]==0.12.1" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
elif ACCEL == "gpu":
    !pip install -q "jax[cuda12]==0.12.1"
else:
    print("WARNING: no TPU/GPU detected - enable an accelerator in Settings; CPU would take days.")

**(4) Backend + license.** Before the first run, accept the Gemma license on Kaggle
(open <https://www.kaggle.com/models/google/gemma-3> and click *Request access*) with the same
account as `KAGGLE_USERNAME`. Otherwise downloading `gemma3_instruct_1b` fails with 403.

In [ ]:
# (4) Keras backend must be set before keras is imported (subprocesses inherit it).
os.environ["KERAS_BACKEND"] = "jax"
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.95")
!python -c "import jax; print(jax.devices())"

In [ ]:
# (5) Build the datasets (downloads one corbt/all-recipes parquet shard + the Ahn et al. flavor data).
!python -m recipe_gemma.data.build --config configs/data.yaml

In [ ]:
# (6) Full LoRA fine-tune (Gemma 3 1B IT, r=16, seq 512). Writes checkpoints/full/adapter.lora.h5,
# results/train_full.json and results/train_full_log.csv.
TRAIN_CONFIG = "configs/train_full.yaml"
!python -m recipe_gemma.train --config {TRAIN_CONFIG}

In [ ]:
# (7) Evaluation + report. configs/eval.yaml decides which adapter/checkpoint is scored; make sure
# it points at the full run (checkpoints/full). The Gemini baseline and the LLM judge need GEMINI_API_KEY;
# without it those rows are reported as "not run".
!python -m recipe_gemma.eval --config configs/eval.yaml
!python -m recipe_gemma.eval.report --config configs/eval.yaml

In [ ]:
# (8) Optional: push the LoRA adapter to the Hugging Face Hub (needs HF_TOKEN with write access).
PUSH_TO_HUB = False
if PUSH_TO_HUB and os.environ.get("HF_TOKEN"):
    !python scripts/push_adapter.py --config {TRAIN_CONFIG}
else:
    print("skipping Hub push (set PUSH_TO_HUB = True and provide HF_TOKEN)")

In [ ]:
# (9) Zip results/ (+ the adapter) for download.
import shutil
from pathlib import Path

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)
print(shutil.make_archive(str(OUT / "recipe_gemma_results"), "zip", root_dir=".", base_dir="results"))
if Path("checkpoints/full").exists():
    print(shutil.make_archive(str(OUT / "recipe_gemma_adapter_full"), "zip", root_dir=".", base_dir="checkpoints/full"))

Download the zips from the **Output** panel (`/kaggle/working`). Then unzip `results/` into
the repo and open `notebooks/results_analysis.ipynb`.